In [60]:
import os, uuid
from functools import lru_cache
from datetime import datetime
from dataclasses import dataclass
from qdrant_client import QdrantClient
from langchain_ollama.embeddings import OllamaEmbeddings
from typing import List, Optional
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from qdrant_client.models import (
    VectorParams, PointStruct, Distance,
    CreateFieldIndex, PayloadSchemaType,
    SparseVector, SparseIndexParams, SparseVectorParams,
    Prefetch, FusionQuery, Fusion
)
from fastembed import SparseTextEmbedding

In [47]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

In [4]:
books_path = os.path.join(os.path.abspath('../'), 'data', 'rag_data')

In [63]:
class RAGVectorStore:
    REQUIRED_ENV_VARS = ["QDRANT_URL", "QDRANT_API_KEY"]
    COLLECTION_NAME = "house_gpt_books"
    SIMILARITY_THRESHOLD = 0.75

    DENSE_VECTOR_NAME = "dense"
    SPARSE_VECTOR_NAME = "sparse"

    def __init__(self) -> None:
        self._validate_env_vars()
        self.dense_model = OllamaEmbeddings(model="qwen3-embedding:8b")
        self.sparse_model = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")
        self.client = QdrantClient(
            url=os.getenv("QDRANT_URL"),
            api_key=os.getenv("QDRANT_API_KEY"),
            timeout=120
        )

    def _validate_env_vars(self) -> None:
        missing = [v for v in self.REQUIRED_ENV_VARS if not os.getenv(v)]
        if missing:
            raise ValueError(f"Missing env vars: {missing}")

    def _collection_exists(self) -> bool:
        collections = self.client.get_collections().collections
        return any(c.name == self.COLLECTION_NAME for c in collections)

    def _create_collection(self) -> None:
        vector_size = len(self.dense_model.embed_query("test"))

        self.client.create_collection(
            collection_name=self.COLLECTION_NAME,
            vectors_config={
                self.DENSE_VECTOR_NAME: VectorParams(
                    size=vector_size,
                    distance=Distance.COSINE
                ),
            },
            sparse_vectors_config={
                self.SPARSE_VECTOR_NAME: SparseVectorParams(
                    index=SparseIndexParams(on_disk=False)
                )
            }
        )

        self.client.create_payload_index(
            collection_name=self.COLLECTION_NAME,
            field_name="book_title",
            field_schema=PayloadSchemaType.KEYWORD
        )

    def _embed_sparse(self, text: str) -> SparseVector:
        result = list(self.sparse_model.embed([text]))[0]
        return SparseVector(indices=result.indices.tolist(), values=result.values.tolist())

    def get_books_data(self, path: str):
        loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyPDFLoader)
        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=50
        )

        split_docs = splitter.split_documents(documents)

        cleaned = []

        for doc in split_docs:
            file_path = doc.metadata.get("source", "")
            book_title = os.path.basename(file_path).replace(".pdf", "")

            cleaned.append({
                "text": doc.page_content,
                "metadata": {
                    "book_title": book_title,
                    "page": doc.metadata.get("page"),
                    "creation_date": doc.metadata.get("creationdate"),
                    "keywords": doc.metadata.get("keywords"),
                    "source": file_path,
                }
            })

        return cleaned

    def store_book(self, text: str, metadata: dict) -> None:
        if not self._collection_exists():
            self._create_collection()

        embedding = self.dense_model.embed_query(text)

        point_id = str(uuid.uuid4())

        payload = {
            "text": text,
            "book_title": metadata.get("book_title"),
            "page": metadata.get("page"),
            "creation_date": metadata.get("creation_date"),
        }

        self.client.upsert(
            collection_name=self.COLLECTION_NAME,
            points=[
                PointStruct(
                    id=point_id,
                    vector=embedding,
                    payload=payload
                )
            ]
        )

    def ingest_books(self, path: str):
        docs = self.get_books_data(path)

        BATCH_SIZE = 100
        points = []

        for doc in docs:
            embedding_dense = self.dense_model.embed_query(doc["text"])
            embedding_sparse = self._embed_sparse(doc["text"])

            points.append(
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector={self.DENSE_VECTOR_NAME: embedding_dense, self.SPARSE_VECTOR_NAME: embedding_sparse},
                    payload={
                        "text": doc["text"],
                        **doc["metadata"]
                    }
                )
            )

        if not self._collection_exists():
            self._create_collection()


        for i in range(0, len(points), BATCH_SIZE):
            batch = points[i:i+BATCH_SIZE]
            self.client.upsert(
                collection_name=self.COLLECTION_NAME,
                points=batch
            )

In [61]:
@lru_cache
def get_rag_vector_store() -> RAGVectorStore:
    return RAGVectorStore()

In [56]:
store = get_rag_vector_store()
store.ingest_books(books_path)

In [57]:
results = store.search_data("heart disease diagnosis")

In [62]:
results

[{'text': 'suggestive of congestive heart failure. heart disease (e.g., troponin, B-type natriuretic \nRales, pedal edema, and jugular peptide) is unclear in children.\nvenous distention commonly seen\n(continued)\nTable 34.1 Differential diagnosis of shortness of breath in children (cont)',
  'book_title': 'clinical_emergency_medicine',
  'metadata': {'page': 528,
   'creation_date': '2006-09-08T13:02:03+02:00',
   'keywords': None,
   'source': '/home/fa-res/Desktop/langgraph/house_gpt/data/rag_data/clinical_emergency_medicine.pdf'},
  'score': 0.5},
 {'text': 'planning an outpatient workup.\nDo you have a previous history of coronary\nartery disease (CAD), congestive heart\nfailure (CHF), dysrhythmias, or valvular \nheart disease? Have you had prior cardiac\nsurgery?\nGiven that CAD, CHF, and primary dysrhyth-\nmias are recurrent illnesses, it is important to\nidentify those patients with a prior history of\nCAD or CHF, as these patients may be more',
  'book_title': 'clinical_emerg